In [36]:
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.messages import AIMessage, SystemMessage, HumanMessage
from langchain_core.output_parsers import StrOutputParser
from dotenv import load_dotenv
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_ollama import OllamaEmbeddings
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

In [37]:
load_dotenv()

True

In [38]:
llm = ChatGroq(model="openai/gpt-oss-120b")

In [39]:
loader = WebBaseLoader("https://www.flipkart.com/apple-iphone-17-white-256-gb/p/itm68ad8410784c7?pid=MOBHQV9YAZYYWY8A&lid=LSTMOBHQV9YAZYYWY8A6D2THC&marketplace=FLIPKART&q=iphone+17&store=tyy%2F4io&srno=s_1_1&otracker=AS_QueryStore_OrganicAutoSuggest_1_6_na_na_ps&otracker1=AS_QueryStore_OrganicAutoSuggest_1_6_na_na_ps&fm=organic&iid=80b9de4e-b925-426a-9fda-e50f9afbba4e.MOBHQV9YAZYYWY8A.SEARCH&ppt=None&ppn=None&ssid=qx5c33qmwvp800sg1790767263369&qH=c9eeb2d6cc488f0b&ov_redirect=true")

In [40]:
docs = loader.load()

In [41]:
docs[0]

Document(metadata={'source': 'https://www.flipkart.com/apple-iphone-17-white-256-gb/p/itm68ad8410784c7?pid=MOBHQV9YAZYYWY8A&lid=LSTMOBHQV9YAZYYWY8A6D2THC&marketplace=FLIPKART&q=iphone+17&store=tyy%2F4io&srno=s_1_1&otracker=AS_QueryStore_OrganicAutoSuggest_1_6_na_na_ps&otracker1=AS_QueryStore_OrganicAutoSuggest_1_6_na_na_ps&fm=organic&iid=80b9de4e-b925-426a-9fda-e50f9afbba4e.MOBHQV9YAZYYWY8A.SEARCH&ppt=None&ppn=None&ssid=qx5c33qmwvp800sg1790767263369&qH=c9eeb2d6cc488f0b&ov_redirect=true', 'title': 'Apple iPhone 17 (256 GB Storage) Online at Best Price On Flipkart.com', 'language': 'en-IN'}, page_content='      Apple iPhone 17 (256 GB Storage) Online at Best Price On Flipkart.com                FlipkartA one-stop ShoppingMinutesEverything in MinutesGroceryAt wholesale pricesTravelAll your travel needsSearch IconSearch IconLoginNew customer?Sign UpMy ProfileFlipkart Plus ZoneOrdersWishlistBecome a SellerRewardsGift CardsNotification Preferences24x7 Customer CareAdvertiseDownload AppMoreMo

In [42]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)

In [43]:
splits = splitter.split_documents(docs)

In [44]:
len(splits)

12

In [45]:
vector_store = Chroma.from_documents(documents=splits, embedding=OllamaEmbeddings(model='nomic-embed-text'))

In [46]:
retriever = vector_store.as_retriever(search_kwargs={'k':2})

In [47]:
system_data='''You are an Helpful Question Answering system .
Answer the query from the provided context. If you dont know the answer directly say Thank you and
I dont know the answer . Keep the answer concise.  '{context}'  '''

In [48]:
prompt = ChatPromptTemplate.from_messages([
    ('system', system_data), 
    ('human', '{input}')
])

In [49]:
question_answer_chain = create_stuff_documents_chain(llm, prompt)

In [50]:
rag_chain = create_retrieval_chain(retriever, question_answer_chain)

In [51]:
response = rag_chain.invoke({'input':'what is the price of iphone 17 ?'})

In [52]:
response['answer']

'The iPhone\u202f17 is priced at\u202f₹95,900.'

In [53]:
response2 = rag_chain.invoke({'input': 'In what colours is it available'})

In [54]:
response2['answer']

"Thank you and I don't know the answer."

In [55]:
response3 = rag_chain.invoke({'input': 'What previous query i asked to you ?'})

In [56]:
response3['answer']

'Thank you and I don’t know the answer.'

In [57]:
## Associating history to llm

In [58]:
from langchain_classic.chains import create_history_aware_retriever
from langchain_core.prompts import MessagesPlaceholder

In [59]:
context_system = '''Given a chat history and the latest user question which might reference context in chat
history the user and dont answer without looking into the chat history'''

In [60]:
context_prompt = ChatPromptTemplate.from_messages([('system','Rephrame the query as a standalone query based on past context or chat history'),
                                              MessagesPlaceholder(variable_name='chat_history'),
                                              ('human', '{input}')])

In [61]:
qa_prompt = ChatPromptTemplate.from_messages([('system',context_system),
                                              MessagesPlaceholder(variable_name='chat_history'),
                                              ('human', '{input} \n\n context: \n {context}')])

In [62]:
# history_aware_retriever = create_history_aware_retriever(llm, retriever, prompt)
history_aware_retriever = create_history_aware_retriever(llm, retriever, context_prompt)

In [63]:
question_answer_chain = create_stuff_documents_chain(llm, qa_prompt)

In [64]:
rag_chain = create_retrieval_chain(history_aware_retriever, question_answer_chain)

In [65]:
chat_history = []

In [66]:
question = 'What is the price of iphone 17'

In [67]:
from operator import itemgetter

In [68]:
result = rag_chain.invoke({'input':question, 'chat_history':chat_history})

In [69]:
chat_history.extend(
    [HumanMessage(content=question),
    AIMessage (content=response['answer'])]
)

In [70]:
question = 'What query i asked you before'

In [73]:
result2 = result = rag_chain.invoke({'input':question, 'chat_history':chat_history})

In [74]:
result2

{'input': 'What query i asked you before',
 'chat_history': [HumanMessage(content='What is the price of iphone 17', additional_kwargs={}, response_metadata={}),
  AIMessage(content='The iPhone\u202f17 is priced at\u202f₹95,900.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])],
 'context': [Document(id='bd50b233-56ae-4e48-8d07-ad839e93f559', metadata={'title': 'Apple iPhone 17 (256 GB Storage) Online at Best Price On Flipkart.com', 'language': 'en-IN', 'source': 'https://www.flipkart.com/apple-iphone-17-white-256-gb/p/itm68ad8410784c7?pid=MOBHQV9YAZYYWY8A&lid=LSTMOBHQV9YAZYYWY8A6D2THC&marketplace=FLIPKART&q=iphone+17&store=tyy%2F4io&srno=s_1_1&otracker=AS_QueryStore_OrganicAutoSuggest_1_6_na_na_ps&otracker1=AS_QueryStore_OrganicAutoSuggest_1_6_na_na_ps&fm=organic&iid=80b9de4e-b925-426a-9fda-e50f9afbba4e.MOBHQV9YAZYYWY8A.SEARCH&ppt=None&ppn=None&ssid=qx5c33qmwvp800sg1790767263369&qH=c9eeb2d6cc488f0b&ov_redirect=true'}, page_content='Apple iPhone 17 (25